
# SisFall CNN TinyML Training Pipeline

This notebook implements:

- Automatic SisFall dataset ingestion
- Fall vs ADL binary classification
- Subject-independent evaluation
- Sliding-window segmentation
- 1D CNN architecture
- TensorFlow Lite Float32 export
- TensorFlow Lite INT8 export
- Edge-device deployment artefacts
- Confusion matrix and performance reports

Recommended dataset layout:

```text
SisFall_dataset/
├── SA01 ... SA23
├── SE01 ... SE15
└── Readme.txt
```


In [ ]:

import os
import re
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    f1_score,
)

from sklearn.preprocessing import StandardScaler
import joblib

import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras import callbacks
from tensorflow.keras import models

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)

print("TensorFlow:", tf.__version__)


In [ ]:

# Configuration

DATASET_ROOT = "SisFall_dataset"

WINDOW_SIZE = 200
STRIDE = 100

TEST_SUBJECT_FRACTION = 0.20

MODEL_DIR = "exports"
os.makedirs(MODEL_DIR, exist_ok=True)

FALL_PREFIX = "F"
ADL_PREFIX = "D"


In [ ]:

# SisFall Loader

def load_sisfall_file(filepath):

    rows = []

    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            values = re.split(r"[,;\s]+", line)

            try:
                values = [float(v) for v in values if v != ""]
            except:
                continue

            rows.append(values)

    if len(rows) == 0:
        return None

    arr = np.asarray(rows, dtype=np.float32)

    return arr


def activity_to_label(activity):

    if activity.startswith(FALL_PREFIX):
        return 1

    return 0


In [ ]:

# Dataset Parsing

samples = []
labels = []
subjects = []

root = Path(DATASET_ROOT)

subject_dirs = sorted(
    [d for d in root.iterdir() if d.is_dir()]
)

for subject_dir in subject_dirs:

    subject = subject_dir.name

    txt_files = sorted(subject_dir.glob("*.txt"))

    for txt_file in txt_files:

        activity = txt_file.stem.split("_")[0]

        label = activity_to_label(activity)

        signal = load_sisfall_file(txt_file)

        if signal is None:
            continue

        for start in range(
            0,
            len(signal) - WINDOW_SIZE,
            STRIDE,
        ):

            window = signal[start:start + WINDOW_SIZE]

            samples.append(window)
            labels.append(label)
            subjects.append(subject)

X = np.asarray(samples, dtype=np.float32)
y = np.asarray(labels, dtype=np.int32)
groups = np.asarray(subjects)

print("Windows:", len(X))
print("Shape:", X.shape)
print("Falls:", np.sum(y == 1))
print("ADL:", np.sum(y == 0))
print("Subjects:", len(np.unique(groups)))


In [ ]:

# Subject-Independent Split

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=TEST_SUBJECT_FRACTION,
    random_state=SEED,
)

train_idx, test_idx = next(
    splitter.split(X, y, groups)
)

X_train = X[train_idx]
X_test = X[test_idx]

y_train = y[train_idx]
y_test = y[test_idx]

train_subjects = np.unique(groups[train_idx])
test_subjects = np.unique(groups[test_idx])

print("Train subjects:", train_subjects)
print("Test subjects:", test_subjects)


In [ ]:

# Channel Normalisation

n_channels = X_train.shape[-1]

scaler = StandardScaler()

X_train_2d = X_train.reshape(-1, n_channels)
X_test_2d = X_test.reshape(-1, n_channels)

X_train_2d = scaler.fit_transform(X_train_2d)
X_test_2d = scaler.transform(X_test_2d)

X_train = X_train_2d.reshape(X_train.shape)
X_test = X_test_2d.reshape(X_test.shape)

joblib.dump(
    scaler,
    f"{MODEL_DIR}/sisfall_scaler.pkl"
)


In [ ]:

# CNN Model

model = models.Sequential([

    layers.Input(
        shape=(
            WINDOW_SIZE,
            X_train.shape[-1]
        )
    ),

    layers.Conv1D(
        32,
        5,
        activation="relu",
        padding="same"
    ),
    layers.BatchNormalization(),
    layers.MaxPooling1D(2),

    layers.Conv1D(
        64,
        5,
        activation="relu",
        padding="same"
    ),
    layers.BatchNormalization(),
    layers.MaxPooling1D(2),

    layers.Conv1D(
        128,
        3,
        activation="relu",
        padding="same"
    ),
    layers.BatchNormalization(),

    layers.GlobalAveragePooling1D(),

    layers.Dense(
        128,
        activation="relu"
    ),
    layers.Dropout(0.3),

    layers.Dense(
        1,
        activation="sigmoid"
    )
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.AUC(name="auc")
    ]
)

model.summary()


In [ ]:

# Training

checkpoint = callbacks.ModelCheckpoint(
    f"{MODEL_DIR}/best_sisfall_cnn.keras",
    save_best_only=True,
    monitor="val_auc",
    mode="max"
)

early_stop = callbacks.EarlyStopping(
    patience=10,
    restore_best_weights=True,
    monitor="val_auc",
    mode="max"
)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=128,
    callbacks=[checkpoint, early_stop],
    verbose=1
)


In [ ]:

# Evaluation

probs = model.predict(X_test).ravel()
preds = (probs >= 0.5).astype(int)

acc = accuracy_score(y_test, preds)
f1 = f1_score(y_test, preds)

print("Accuracy:", acc)
print("F1:", f1)

print(
    classification_report(
        y_test,
        preds,
        target_names=[
            "ADL",
            "FALL"
        ]
    )
)


In [ ]:

# Confusion Matrix

cm = confusion_matrix(y_test, preds)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["ADL","FALL"]
)

disp.plot()
plt.show()


In [ ]:

# Save Keras Model

model.save(
    f"{MODEL_DIR}/sisfall_cnn.keras"
)


In [ ]:

# Float32 TFLite Export

converter = tf.lite.TFLiteConverter.from_keras_model(model)

tflite_model = converter.convert()

with open(
    f"{MODEL_DIR}/sisfall_float32.tflite",
    "wb"
) as f:
    f.write(tflite_model)

print("Float32 export complete")


In [ ]:

# INT8 Quantisation

def representative_dataset():

    for i in range(
        min(
            1000,
            len(X_train)
        )
    ):
        yield [X_train[i:i+1].astype(np.float32)]

converter = tf.lite.TFLiteConverter.from_keras_model(model)

converter.optimizations = [
    tf.lite.Optimize.DEFAULT
]

converter.representative_dataset = representative_dataset

converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS_INT8
]

converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

quant_model = converter.convert()

with open(
    f"{MODEL_DIR}/sisfall_int8.tflite",
    "wb"
) as f:
    f.write(quant_model)

print("INT8 export complete")


In [ ]:

# TFLite Validation

interpreter = tf.lite.Interpreter(
    model_path=f"{MODEL_DIR}/sisfall_int8.tflite"
)

interpreter.allocate_tensors()

print(
    interpreter.get_input_details()
)
print(
    interpreter.get_output_details()
)
